This notebook is for analyzing H$\alpha$ in the WASP-12b_20201122_PEPSI dataset.

# Imports

In [9]:
import os
os.chdir('/home/paiasnodkar.1/AtmosphericDynamics/')
import glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import astropy
import astropy.io.fits as FITS
from astropy.coordinates import Angle
from astropy import units as u

try:
    import cPickle as pickle
except ModuleNotFoundError:
    import pickle

import TransmissionSpectroscopy_lib as TSL

import importlib
importlib.reload(TSL)

plt.rcParams['pdf.fonttype'] = 42
plt.rc("font", **{"sans-serif": ["Roboto"], "size": 20})
plt.rc("text", usetex = True)

plt.rcParams['font.family'] = 'sans-serif'

# System properties

In [2]:
planet_name = 'WASP-12b_20201122_PEPSI'
planet_name_ref = 'WASP-12b'

# Ephemerides (days)
t0, t0_err = 2456176.668258, 7.7650773e-5
P, P_err = 1.0914203, 1.4432653e-7
T14, T14_err_pos, T14_err_neg = 0.12483, 0.00044, 0.00043 # "total transit duration" 
tau, tau_err =  0.01526, 0.00044 # "ingress/egress duration"

# Read in observations

In [18]:
with open('data/'+planet_name+'.pkl', "rb") as input_file:
    FM = pickle.load(input_file)

# Initial code snippets to read data

Read in PEPSI observations and print JD_UTC times.

In [3]:
data_dir = 'data/WASP-12b_20201122_PEPSI/pepsib.20201122.*.dxt.nor'
dirFiles = glob.glob(data_dir)
dirFiles.sort(key=lambda f: int(''.join(filter(str.isdigit, f))))

# Read first file
spec_file = FITS.open(dirFiles[0])
times_JDUTC = [spec_file[0].header['JD-OBS']]

print('RA:', spec_file[0].header['RA'], 'HH:MM:SS')
print('Dec:', spec_file[0].header['DEC'], 'DD:MM:SS')
latitude = Angle(spec_file[0].header['LATITUDE'] + ' degrees')
latitude = float(latitude.to_string(decimal=True))
longitude = Angle(spec_file[0].header['LONGITUD'], unit='hourangle')
longitude = float(longitude.to_string(unit=u.deg, decimal=True))
print('Latitude:', latitude, 'degrees')
print('Longitude:', longitude, 'degrees')
print('Altitude:', spec_file[0].header['ALTITUDE'], 'm \n')


# Read in all other files
for file in dirFiles[1:]:
    spec_file = FITS.open(file)
    times_JDUTC.append(spec_file[0].header['JD-OBS'])

[print(t) for t in times_JDUTC];

RA: 06:31:52.79 HH:MM:SS
Dec: +29:39:23.30 DD:MM:SS
Latitude: 32.7013 degrees
Longitude: -109.889 degrees
Altitude: 3221 m 

2459175.80564815
2459175.81660995
2459175.82756714
2459175.83852661
2459175.84948495
2459175.86044097
2459175.87140046
2459175.88235763
2459175.89331597
2459175.90427314
2459175.91522918
2459175.92618634
2459175.93714469
2459175.94810303
2459175.95906132
2459175.97001736
2459175.98097453
2459175.9919317
2459176.00288773
2459176.0138449
2459176.02480207
2459176.03575928
2459176.04671527
2459176.05737978


Create FluxMap object.

In [8]:
times_BJDTDB = np.array(pd.read_csv('data/'+planet_name+'/'+planet_name+'_times_BJDTDB.txt', header=None)).flatten()

spec_file = FITS.open(file)
wav_spec = spec_file[1].data["Arg"]
flx_spec = spec_file[1].data["Fun"]
flx_spec_err = np.sqrt(spec_file[1].data["Var"])
FM = TSL.FluxMap(wav_spec, times_BJDTDB[0], flx_spec, flx_spec_err, t0, t0_err, P, P_err, T14, tau,
                 time_system='BJD_TDB', spec_type=False)

for i, file in enumerate(dirFiles[1:]):
    spec_file = FITS.open(file)
    wav_spec = spec_file[1].data["Arg"]
    flx_spec = spec_file[1].data["Fun"]
    flx_spec_err = np.sqrt(spec_file[1].data["Var"])
    FM.addNewObservation(wav_spec, times_BJDTDB[i+1], flx_spec, flx_spec_err)
    
TSL.save_object(FM, 'data/'+planet_name+'.pkl')